## Country Code 


In [0]:
import dlt
from pyspark.sql.functions import *

### Streaming Table 

In [0]:
import dlt
import pyspark.sql.functions as F

rules = {
    "valid_id":           "id IS NOT NULL",
    "valid_country_code": "country_code IS NOT NULL",   # this table's column, not activity_code
    "valid_sequence":     "created_at IS NOT NULL",
}


@dlt.view(name="DimCountryCode_Stage")
@dlt.expect_all_or_drop(rules)
def DimCountryCode_Stage():
    return spark.readStream.table("travel_journal_catalog.silver.country_code")


dlt.create_streaming_table("dim_country_code")

dlt.apply_changes(
    target             = "dim_country_code",
    source             = "DimCountryCode_Stage",
    keys               = ["id"],
    sequence_by        = "created_at",
    stored_as_scd_type = 1,              # int, not "1"
    except_column_list = ["flag"],
)